# 4.10 本地模型部署实战(Ollama + QwQ-32B)

对应笔记 **4.7.3 部署本地模型**。

前面几节用的都是**在线模型**(Qwen,走阿里云 API)。本节演示如何在本地部署一个大模型,并用 OpenAI 兼容接口调用它:

1. **环境准备** —— 安装 Ollama;
2. **下载模型** —— `ollama pull` 拉取 QwQ-32B;
3. **启动服务** —— `ollama serve` 常驻本地;
4. **连接本地模型** —— 用 OpenAI 客户端调 `localhost:11434`;
5. **进阶** —— 把 4.8 的 Function calling 也接到本地模型上。

> 为什么用 Ollama 而不是 vLLM?
> - **vLLM 不支持原生 Windows**,只能在 WSL2/Linux 跑,而且需要 CUDA + 编译环境,配置门槛高;
> - **Ollama 原生支持 Windows**,一条命令装好即用,自动处理显存/内存调度、GPU 加速;
> - 你机器是 **Windows + RTX 5060 Ti(16GB 显存)**,Ollama 是当前最现实的选择。
>
> ⚠️ QwQ-32B 完整版约 24GB 参数,16GB 显存放不下,Ollama 会**自动把超出部分放到内存**(32GB)里跑,速度慢但能用。

## 0. 环境准备

```bash
# 下载安装 Ollama(Windows 原生,免 WSL2)
# 官网: https://ollama.com/download
# 或: winget install Ollama.Ollama
```
> 一句话总结:Ollama 是跨平台的本地大模型运行器,Windows 原生支持,装好后自带 OpenAI 兼容接口。

In [ ]:
# 检查环境:Ollama 是否已装、服务是否在跑
import subprocess
import urllib.request
import json

# 1) 检查 ollama 命令
try:
    ver = subprocess.run(["ollama", "--version"], capture_output=True, text=True, timeout=10)
    print("✅ ollama 客户端:", ver.stdout.strip().split("\n")[0])
except FileNotFoundError:
    print("❌ ollama 未安装,请先到 https://ollama.com/download 安装")
    raise SystemExit

# 2) 检查服务是否在跑(默认端口 11434)
try:
    with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3) as r:
        data = json.loads(r.read())
        models = [m["name"] for m in data.get("models", [])]
        print("✅ Ollama 服务运行中,已安装模型:", models or "(暂无)")
except Exception:
    print("⚠️ Ollama 服务未运行,请执行: ollama serve")

## 1. 下载模型(Ollama)

```bash
# 拉取 QwQ-32B(默认 Q4 量化版,约 20GB)
ollama pull qwq
```
> 一句话总结:一条命令把 QwQ-32B 拉到本地,以后 `ollama run qwq` 随时可用。
> ⚠️ 首次下载约 20GB,耗时长;下载完以后离线也能用。

In [ ]:
# 检查模型是否已拉取
import urllib.request, json

try:
    with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3) as r:
        data = json.loads(r.read())
        models = [m["name"] for m in data.get("models", [])]
    if any("qwq" in m for m in models):
        print("✅ QwQ-32B 已就绪:", models)
    else:
        print("⏳ 还没拉 qwq,请先在终端执行: ollama pull qwq")
except Exception:
    print("⚠️ 服务未运行")

## 2. 启动 Ollama 服务

Ollama 安装后服务默认**开机自启**(Windows 上以系统服务运行)。

```bash
# 如果服务没跑,手动启动(常驻,会一直占用终端):
ollama serve

# 模型已下载后,可以直接在终端对话:
ollama run qwq
```
> 一句话总结:Ollama 服务默认常驻 `localhost:11434`,提供 OpenAI 兼容接口,`ollama run qwq` 是它自带的命令行聊天入口。

## 3. 用 OpenAI 客户端调用本地模型

Ollama 暴露 **OpenAI 兼容接口**(`http://localhost:11434/v1`),所以客户端代码和 4.8/4.9 几乎一样,只需改:
- `base_url` → `http://localhost:11434/v1`
- `api_key` → `"EMPTY"`(本地服务不校验 key)

In [ ]:
from openai import OpenAI

openai_api_key = "EMPTY"  # 本地服务不校验 key
openai_api_base = "http://localhost:11434/v1"  # Ollama 的 OpenAI 兼容接口

client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)

prompt = "在单词\"strawberry\"中，总共有几个R？"
messages = [
    {"role": "user", "content": prompt}
]
response = client.chat.completions.create(
    model="qwq",
    messages=messages,
)

print(response.choices[0].message.content)

## 4. 连续对话(多轮)

本地模型同样支持多轮对话——把历史消息都传进去,模型就能"记住"上下文。

In [ ]:
conversation = [
    {"role": "system", "content": "你是一个乐于助人的助手,回答要简洁。"},
    {"role": "user", "content": "什么是 MCP?"},
]
r1 = client.chat.completions.create(model="qwq", messages=conversation)
a1 = r1.choices[0].message.content
print("用户: 什么是 MCP?")
print("模型:", a1, "\n")

# 追问,带上历史
conversation.append({"role": "assistant", "content": a1})
conversation.append({"role": "user", "content": "那它和 Function calling 有什么区别?"})
r2 = client.chat.completions.create(model="qwq", messages=conversation)
print("用户: 那它和 Function calling 有什么区别?")
print("模型:", r2.choices[0].message.content)

## 5. 进阶:把 Function calling 接到本地模型

Ollama 同样支持 OpenAI 的 `tools` 参数,4.8 的 Function calling 循环能直接用——只需把 `client` 换成连本地服务的那个。

In [ ]:
# 复用 4.8 的天气函数(本地模拟数据源)
WEATHER_DB = {
    "北京": {"weather": "晴", "temperature": 32},
    "杭州": {"weather": "小雨", "temperature": 27},
    "上海": {"weather": "多云", "temperature": 30},
}

def get_current_weather(location: str) -> str:
    key = location.rstrip("市")
    data = WEATHER_DB.get(key, {"weather": "未知", "temperature": None})
    return f"城市:{location},天气:{data['weather']},气温:{data['temperature']}℃"

# 工具定义(JSON Schema)
tools = [{
    "type": "function",
    "function": {
        "name": "get_current_weather",
        "description": "当你想查询指定城市的天气时非常有用。",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "城市或县区,比如北京市、杭州市、余杭区等。"}
            },
            "required": ["location"],
        },
    },
}]

# 第一轮:让模型决定是否调用工具
resp = client.chat.completions.create(
    model="qwq",
    messages=[{"role": "user", "content": "北京今天天气怎么样?"}],
    tools=tools,
    tool_choice="auto",
)
msg = resp.choices[0].message
print("模型是否想调工具:", bool(msg.tool_calls))

if msg.tool_calls:
    # 执行工具
    import json
    for tc in msg.tool_calls:
        args = json.loads(tc.function.arguments)
        result = get_current_weather(args["location"])
        print(f"调用 {tc.function.name}{args} -> {result}")

        # 把工具结果回传给模型,生成最终答案
        resp2 = client.chat.completions.create(
            model="qwq",
            messages=[
                {"role": "user", "content": "北京今天天气怎么样?"},
                msg,
                {"role": "tool", "tool_call_id": tc.id, "content": result},
            ],
            tools=tools,
        )
        print("最终答案:", resp2.choices[0].message.content)
else:
    print("最终答案:", msg.content)

## 6. 换成真实 vLLM(可选,Linux/WSL 场景)

如果你有 Linux 服务器或 WSL2 环境,也可以用 vLLM 跑同样的模型:

```bash
# 在 Linux/WSL2 里:
pip install vllm
vllm serve ./QwQ-32B --max-model-len 32768   # 单卡 32k
```

客户端只需改 `base_url`:
```python
client = OpenAI(
    api_key="EMPTY",
    base_url="http://localhost:8000/v1",   # vLLM 默认端口
)
```

> 一句话总结:Ollama 与 vLLM 都提供 OpenAI 兼容接口,换后端只改 `base_url` 和 `model` 名,客户端代码完全复用。

## 📊 总结

### 本地部署 vs 在线模型

| 对比项 | 在线 API(Qwen) | 本地 Ollama(QwQ-32B) |
|-------|---------------|---------------------|
| 网络 | 依赖外网,可能不稳定 | 本地推理,数据不出机器 |
| 成本 | 按调用量计费 | 一次性硬件投入(需 GPU) |
| 隐私 | 数据发给云端 | 数据完全本地 |
| 接口 | OpenAI 兼容 | OpenAI 兼容(**代码几乎不用改**) |
| 环境要求 | 只需 API Key | 16GB 显存 + 32GB 内存(量化版) |

### 核心链路

```
ollama pull qwq(下载模型) → ollama serve(启动,默认 11434)
                                    ↓
              localhost:11434/v1 (OpenAI 兼容)
                                    ↓
           OpenAI 客户端 base_url 指向它,其余照旧
```

### 为什么用 Ollama 而不是 vLLM
- vLLM **不支持原生 Windows**(只支持 Linux/WSL2),配置门槛高;
- Ollama **原生支持 Windows**,一条命令装好即用,自动调度显存/内存;
- 两者接口兼容,后端可平滑切换。

### 一句话总结
> 本地部署就是把"在线 API 的 base_url"换成自己机器上的 Ollama 服务——`ollama pull` 下载模型,`ollama serve` 提供接口,客户端代码和前面学的 Function calling / MCP 完全复用。